# Phase 2 — first end-to-end run (BasicTS 1.0, STID, PEMS04)
Run the cells **one by one** (Shift+Enter). Goal: prove the pipeline works and see the real output format.
Colab forgets everything when the session closes: code lives on GitHub, PEMS data is cached in your Drive (`proposal_data`).
Cells 1-5 need no GPU. Cell 6 has a switch: `GPUS = "none"` (CPU) or `"0"` (GPU).

## 1. Which hardware do I have?

In [ ]:
!nvidia-smi 2>&1 | head -15
import sys
print(sys.version)

## 2. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the repo and build an isolated Python 3.11 environment (takes a few minutes)
Colab runs Python 3.13, but BasicTS pins old packages (e.g. numpy 1.24.4) that cannot be built on 3.13.
If this session already has the repo / environment, they are reused.

In [ ]:
import os
%cd /content
if os.path.isdir("/content/proposal/.git"):
    !cd /content/proposal && git pull
else:
    !git clone --recursive https://github.com/hrfpour/proposal.git
%cd /content/proposal/code/BasicTS
if not os.path.exists("/content/venv/bin/python"):
    !pip -q install uv
    !uv venv /content/venv --python 3.11
    !uv pip install --python /content/venv/bin/python -r requirements.txt 2>&1 | tail -n 25
else:
    print("venv already exists - skipping install")

## 4. Make sure PyTorch exists in that environment, and that BasicTS imports

In [ ]:
import subprocess
PY = "/content/venv/bin/python"
if subprocess.run([PY, "-c", "import torch"], capture_output=True).returncode != 0:
    print("torch not in requirements.txt -> installing it")
    !uv pip install --python {PY} torch
!{PY} -c "import sys; sys.path.insert(0, '/content/proposal/code/BasicTS/src'); import basicts, torch, numpy; print('basicts', basicts.__version__, '| torch', torch.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"

## 5. Put the PEMS04 / PEMS08 data in place
Uses the PEMS-only copy cached in your Drive (`MyDrive/proposal_data`). Only if that cache is missing, it downloads `datasets.zip` with the link in `DATA_URL`.

In [ ]:
import os, glob, shutil
KEEP = ["PEMS04", "PEMS08"]
CACHE = "/content/drive/MyDrive/proposal_data"
LOCAL_ZIP = "/content/datasets.zip"
DATA_URL = "PASTE_YOUR_LINK_HERE"   # only needed if the cache is missing
BT = "/content/proposal/code/BasicTS"
os.makedirs(f"{BT}/datasets", exist_ok=True)

if not all(os.path.isdir(f"{CACHE}/{n}") for n in KEEP):
    print("Cache missing -> downloading datasets.zip")
    assert DATA_URL.startswith("http"), "Edit DATA_URL first"
    !pip -q install gdown
    !gdown --fuzzy "{DATA_URL}" -O {LOCAL_ZIP}
    !unzip -l {LOCAL_ZIP} | grep -i -E "PEMS0[48]" | head -20
    !rm -rf /content/_tmp && mkdir -p /content/_tmp
    !unzip -q -o {LOCAL_ZIP} '*PEMS04*' '*PEMS08*' -d /content/_tmp
    os.makedirs(CACHE, exist_ok=True)
    for n in KEEP:
        hits = [p for p in glob.glob(f"/content/_tmp/**/{n}", recursive=True) if os.path.isdir(p)]
        print(n, "->", hits)
        assert hits, f"Folder {n} not found inside the zip"
        shutil.copytree(hits[0], f"{CACHE}/{n}", dirs_exist_ok=True)

for n in KEEP:
    shutil.copytree(f"{CACHE}/{n}", f"{BT}/datasets/{n}", dirs_exist_ok=True)
!ls -la {BT}/datasets/PEMS04

## 6. Train STID on PEMS04 (pipeline check)
`GPUS = "none"` runs on CPU (no GPU quota needed). If your runtime has a GPU, set `GPUS = "0"`.
The script reads the dataset's own settings from `meta.json` (original-scale metrics, zeros masked).

In [ ]:
GPUS = "none"
EPOCHS = 2
PY = "/content/venv/bin/python"
!{PY} /content/proposal/code/scripts/run_stid.py --dataset PEMS04 --epochs {EPOCHS} --gpus {GPUS} 2>&1 | tee /content/run.log | tail -n 3

## 7. Show what the run printed and produced (send this output back)

In [ ]:
print("========== HEAD OF LOG ==========")
!head -n 80 /content/run.log
print("\n========== TAIL OF LOG ==========")
!tail -n 60 /content/run.log
print("\n========== FILES IN checkpoints ==========")
!find /content/proposal/code/BasicTS/checkpoints -type f | head -30

## 8. Export the result JSON and copy it to Drive

In [ ]:
EXPORT_STATUS = "smoke_test"   # keep as is for this pipeline check
DEVICE = "cpu" if GPUS == "none" else "gpu"
!python /content/proposal/code/scripts/export_result.py --ckpt-root /content/proposal/code/BasicTS/checkpoints --model STID --dataset PEMS04 --seed 42 --epochs {EPOCHS} --status {EXPORT_STATUS} --device {DEVICE} --out /content/results_out
!ls /content/results_out
import shutil, time
dest = f"/content/drive/MyDrive/proposal_runs/{time.strftime('%Y%m%d_%H%M')}"
shutil.copytree("/content/results_out", dest)
print("Saved to Drive:", dest)